# 05 — Evaluación final

Evalúa cuatro comportamientos representativos del MVP usando el backend cloud reproducible.


In [ ]:
from pathlib import Path
import pandas as pd
from src.corpus_builder import build_complete_corpus
from src.cloud_backend import CloudRAGPipeline

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

records = build_complete_corpus(ROOT / "data", include_sample=True)
pipe = CloudRAGPipeline(
    dense_min_score=0.45,
    top_k=5,
    temperature=0.0,
).fit(records)


In [ ]:
questions = [
    "¿Cuál es el stock del taladro DEWALT modelo DWD024-B3?",
    "¿Cuántas impresoras 3D Creality K1 están disponibles?",
    "¿Qué componentes principales integra la arquitectura del Simulador Táctico Submarino?",
    "¿Cuál es el número de serie del generador principal de la fragata inexistente?",
]

rows = []
for q in questions:
    r = pipe.ask(q)
    rows.append({
        "question": q,
        "mode": r.response_mode,
        "abstained": r.abstained,
        "stage": r.abstention_stage,
        "sources": r.retrieval_count,
        "answer": r.answer,
    })

result = pd.DataFrame(rows)
display(result)


In [ ]:
assert result.loc[0, "mode"] == "structured"
assert result.loc[1, "abstained"] == False
assert "1 unidad(es)" in result.loc[1, "answer"]
assert result.loc[2, "mode"] == "rag"
assert result.loc[3, "abstained"] == True

print("EVALUACIÓN FUNCIONAL: OK")


## Conclusión

La evaluación final comprueba:
- consultas estructuradas;
- RAG cuantitativo sample;
- RAG documental;
- abstención responsable.

Estas pruebas son de aceptación funcional sobre el corpus académico y no sustituyen una validación operacional con datos institucionales reales.
